In [59]:
# ==========================
# INSTALL
# ==========================

!pip -q install transformers gradio

# ==========================
# IMPORT
# ==========================

import torch
import gradio as gr
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ==========================
# LOAD MODEL
# ==========================

model_path = "/content/drive/MyDrive/NLP/distilbert_clickbait_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

print("✅ DistilBERT Loaded Successfully")

print()

print("Layers :", model.config.n_layers)

print("Hidden Size :", model.config.hidden_size)

print("Heads :", model.config.n_heads)

print("Vocabulary :", tokenizer.vocab_size)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ DistilBERT Loaded Successfully

Layers : 6
Hidden Size : 768
Heads : 12
Vocabulary : 30522


In [81]:
import torch
import numpy as np

def analyze(sentence):

    # ======================================================
    # 1. TOKENIZATION
    # ======================================================

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    with torch.no_grad():

        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )

    # ======================================================
    # 2. TOKEN INFORMATION
    # ======================================================

    input_ids = inputs["input_ids"][0]

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    ids = input_ids.tolist()

    # ======================================================
    # 3. HIDDEN STATES
    #
    # hidden_states[0] : Embedding
    # hidden_states[1] : Layer1
    # ...
    # hidden_states[6] : Layer6
    # ======================================================

    hidden_states = outputs.hidden_states

    layers = []

    for i, hidden in enumerate(hidden_states):

        hidden = hidden.squeeze(0)

        if i == 0:
            layer_name = "Embedding"
        else:
            layer_name = f"Transformer Layer {i}"

        token_details = []

        for token, vector in zip(tokens, hidden):

            token_details.append({

                "token": token,

                "norm": round(
                    torch.norm(vector).item(),
                    3
                ),

                "preview": np.round(
                    vector[:8].cpu().numpy(),
                    3
                ).tolist()

            })

        cls_vector = hidden[0]

        layers.append({

            "name": layer_name,

            "shape": tuple(hidden.shape),

            "cls_vector": cls_vector.cpu().numpy(),

            "cls_norm": round(
                torch.norm(cls_vector).item(),
                3
            ),

            "tokens": token_details

        })

    # ======================================================
    # 4. CLASSIFICATION
    # ======================================================

    logits = outputs.logits.squeeze(0)

    probabilities = torch.softmax(
        logits,
        dim=0
    )

    prediction_index = torch.argmax(
        probabilities
    ).item()

    # Nếu model có id2label thì dùng luôn
    if hasattr(model.config, "id2label"):
        prediction_label = model.config.id2label[prediction_index]
    else:
        prediction_label = (
            "Clickbait"
            if prediction_index == 1
            else "Non-clickbait"
        )

    result = {

        "sentence": sentence,

        "tokens": tokens,

        "input_ids": ids,

        "layers": layers,

        "logits": {

            "Non-clickbait": round(
                float(logits[0]),3
            ),

            "Clickbait": round(
                float(logits[1]),3
            )

        },

        "softmax": {

            "Non-clickbait": round(
                float(probabilities[0]),4
            ),

            "Clickbait": round(
                float(probabilities[1]),4
            )

        },

        "prediction_index": prediction_index,

        "prediction": prediction_label,

        "confidence": round(
            float(probabilities[prediction_index])*100,
            2
        )

    }

    return result

In [82]:
def build_html(sentence):

    result = analyze(sentence)

    # ==========================
    # TOKEN
    # ==========================

    token_html = ""

    for token in result["tokens"]:

        token_html += f"""
        <span style="
            display:inline-block;
            padding:6px;
            margin:3px;
            border:1px solid #666;
            border-radius:5px;
            font-family:monospace;
        ">
        {token}
        </span>
        """

    # ==========================
    # INPUT IDS
    # ==========================

    id_html = ""

    for idx in result["input_ids"]:

        id_html += f"""
        <span style="
            display:inline-block;
            padding:5px;
            margin:3px;
            border:1px solid gray;
            border-radius:5px;
            font-family:monospace;
        ">
        {idx}
        </span>
        """

    # ==========================
    # TRACK TOKEN
    # ==========================

    token_index = 1 if len(result["tokens"]) > 2 else 0

    tracked = result["tokens"][token_index]

    transformer_rows = ""

    for layer in result["layers"]:

        info = layer["tokens"][token_index]

        transformer_rows += f"""
        <tr>

        <td>{layer["name"]}</td>

        <td>{info["norm"]}</td>

        <td>{info["preview"]}</td>

        </tr>
        """

    # ==========================
    # CLS
    # ==========================

    cls_rows = ""

    for layer in result["layers"]:

        cls_rows += f"""
        <tr>

        <td>{layer["name"]}</td>

        <td>{layer["cls_norm"]}</td>

        </tr>
        """

    # ==========================
    # LOGITS
    # ==========================

    logits = result["logits"]

    probs = result["softmax"]

    if probs["Clickbait"] > probs["Non-clickbait"]:

        final_class = "Clickbait"

    else:

        final_class = "Non-clickbait"

    # ==========================
    # HTML
    # ==========================

    html = f"""

    <h1 align='center'>

    DistilBERT Processing Visualizer

    </h1>

    <hr>

    <h2>STEP 1. Input Sentence</h2>

    <p>{sentence}</p>

    <hr>

    <h2>STEP 2. Tokenization</h2>

    {token_html}

    <br><br>

    <b>Input IDs</b>

    <br><br>

    {id_html}

    <hr>

    <h2>STEP 3. Embedding</h2>

    Each token is converted into a
    <b>768-dimensional vector</b>.

    <br><br>

    Embedding Matrix Shape :

    <b>{result["layers"][0]["shape"]}</b>

    <hr>

    <h2>STEP 4. Transformer (Context Learning)</h2>

    <p>

    Tracking token :

    <b>{tracked}</b>

    </p>

    <table border="1" cellpadding="6">

    <tr>

    <th>Layer</th>

    <th>Vector Norm</th>

    <th>First 8 Dimensions</th>

    </tr>

    {transformer_rows}

    </table>

    <br>

    <p>

    ✓ Each Transformer layer updates the vector of every token
    by learning contextual information from surrounding words.

    </p>

    <hr>

    <h2>STEP 5. CLS Vector</h2>

    <p>

    After the final Transformer layer,
    the <b>[CLS]</b> token becomes the representation
    of the whole sentence.

    </p>

    <table border="1" cellpadding="6">

    <tr>

    <th>Layer</th>

    <th>CLS Vector Norm</th>

    </tr>

    {cls_rows}

    </table>

    <hr>

    <h2>STEP 6. Classification Head</h2>

    <p>

    The CLS vector is passed through a Linear layer,
    producing two scores called <b>Logits</b>.

    </p>

    <table border="1" cellpadding="8">

    <tr>

    <th>Class</th>

    <th>Logit</th>

    <th>Softmax Probability</th>

    </tr>

    <tr>

    <td>Non-clickbait</td>

    <td>{logits["Non-clickbait"]}</td>

    <td>{probs["Non-clickbait"]*100:.2f}%</td>

    </tr>

    <tr>

    <td>Clickbait</td>

    <td>{logits["Clickbait"]}</td>

    <td>{probs["Clickbait"]*100:.2f}%</td>

    </tr>

    </table>

    <br>

    <p>

    <b>Argmax</b> selects the class with the highest probability.

    </p>

    <hr>

    <h2>FINAL PREDICTION</h2>

    <h1>

    {final_class}

    </h1>

    <h2>

    Confidence : {result["confidence"]}%

    </h2>

    """

    return html

In [83]:
import gradio as gr


def run_demo(sentence):

    if sentence.strip() == "":
        return """
        <h2>Please enter an English headline.</h2>
        """

    return build_html(sentence)


with gr.Blocks(
    title="DistilBERT Processing Visualizer"
) as demo:

    gr.Markdown(
        """
# DistilBERT Processing Visualizer

This demo visualizes how DistilBERT processes an English headline,
from tokenization to the final prediction.

---
"""
    )

    sentence = gr.Textbox(

        label="English Headline",

        placeholder="Example: You won't believe what happened next...",

        lines=2

    )

    analyze_btn = gr.Button(

        "Analyze",

        variant="primary"

    )

    output = gr.HTML()

    analyze_btn.click(

        fn=run_demo,

        inputs=sentence,

        outputs=output

    )

    gr.Examples(

        examples=[

            ["You won't believe what happened next"],

            ["Scientists discover new species in Amazon rainforest"],

            ["Apple introduces new AI features"],

            ["10 things every student should know"],

            ["The secret doctors don't want you to know"]

        ],

        inputs=sentence

    )

In [84]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7cd14d5d80dcfd14bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://7cd14d5d80dcfd14bf.gradio.live


In [ ]:
//clickbait
The Unbelievable Stats Behind How Cristiano Ronaldo Scored 50 Goals at Age 41
You Won’t Believe How This New OpenAI Model Passed the Bar Exam with a Perfect Score
This Secret Feature On Your iPhone Is Draining Your Battery Lifespan
This Popular Social Media App Is Quietly Accessing Your Camera In The Background
Five everyday foods that are slowly damaging your brain health
New Study Reveals The Secret Behind Better Sleep
Scientists Discover Something In The Amazon That Will Shock You
//non-clickbait
Inside the Secret AI Algorithm That Boosted Amazon’s Sales by 40% Overnight
How a Single Line of Code Cost a Major Crypto Exchange $500 Million in Less Than 5 Minutes
"Is the Era of Traditional Software Engineers Officially Over? What the Data Suggests"
Scientists discover new species in Amazon rainforest

